# AULA 20/10 — REVISÃO ORIENTADA PARA O MINI-PROJETO M1.1
## Sprint 1: leitura, diagnóstico e qualidade dos dados

### Objetivo da aula
Hoje você vai praticar **as primeiras etapas obrigatórias do mini-projeto**, mas usando uma base de treino diferente da base avaliativa.

Ao final, você deverá saber responder:

1. Quantas linhas e colunas a base possui?
2. O que cada linha representa?
3. Quais são os tipos de dados?
4. Existem valores ausentes?
5. Existem textos vazios ou apenas com espaços?
6. Existem duplicatas exatas?
7. Existem categorias escritas de maneiras diferentes?

> **Importante:** nesta aula não queremos “limpar tudo rapidamente”. Primeiro, queremos aprender a **diagnosticar e registrar o problema**.

## Roteiro de 3 horas

- **0:00–0:20** — Entendendo a proposta avaliativa e os critérios.
- **0:20–0:45** — Leitura do CSV e interpretação da unidade de análise.
- **0:45–1:15** — `head()`, `shape`, `columns`, `info()` e tipos.
- **1:15–1:35** — Valores ausentes e textos vazios.
- **1:35–1:50** — Intervalo.
- **1:50–2:15** — Duplicatas e inconsistências textuais.
- **2:15–2:40** — Registro do diagnóstico antes da limpeza.
- **2:40–3:00** — Desafio de auditoria + checklist da Sprint 1.

## ETAPA 1 — Carregando a base

### Por que começamos aqui?
No mini-projeto, o código precisa conseguir **reproduzir a análise do início ao fim**. Isso significa que a leitura do arquivo deve aparecer claramente no notebook.

Faça upload de `pedidos_diagnostico.csv`.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving pedidos_diagnostico.csv to pedidos_diagnostico.csv


In [2]:
import pandas as pd

df = pd.read_csv("pedidos_diagnostico.csv")

df_original = df.copy()

display(df.head())

,pedido_id,cliente,data_pedido,categoria,valor_compra,quantidade,canal,status
0,P0001,Cliente 01,2026-09-25,Beleza,499.33,5.0,Loja,pendente
1,P0002,Cliente 02,2026-09-20,Moda,654.57,1.0,App,entregue
2,P0003,Cliente 03,2026-09-03,eletronicos,496.24,4.0,Site,entregue
3,P0004,Cliente 04,2026-09-25,Beleza,685.40,3.0,Site,entregue
4,P0005,NaN,2026-09-20,Beleza,157.59,2.0,Loja,entregue


### Pergunta didática
**O que cada linha representa?**

Observe as colunas. Neste exercício, cada linha representa **um pedido**.

Isso importa porque:
- contar linhas = contar pedidos;
- somar `valor_compra` = somar valores dos pedidos;
- contar clientes únicos é outra pergunta;
- não devemos chamar qualquer contagem de “quantidade de clientes”.

## ETAPA 2 — Estrutura da base - Investigação de estrutura e tipos

In [3]:
# Quantidade de linhas e colunas
print("Quantidade de linhas: ", df.shape[0])
print("quantidade de colunas: ", df.shape[1])

Quantidade de linhas:  38
quantidade de colunas:  8


In [5]:
# Nomes das colunas
print(df.columns.tolist())

['pedido_id', 'cliente', 'data_pedido', 'categoria', 'valor_compra', 'quantidade', 'canal', 'status']


In [6]:
# Tipos e quantidade de valores preenchidos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   pedido_id     38 non-null     object 
 1   cliente       37 non-null     object 
 2   data_pedido   38 non-null     object 
 3   categoria     38 non-null     object 
 4   valor_compra  36 non-null     float64
 5   quantidade    37 non-null     float64
 6   canal         38 non-null     object 
 7   status        38 non-null     object 
dtypes: float64(2), object(6)
memory usage: 2.5+ KB


### O que registrar no relatório?
Você deve ser capaz de escrever algo como:

> “A base possui 38 linhas e 8 colunas. Cada linha representa um valor presente. As principais colunas usadas na análise são pedido_id, cliente, data_pedido, categoria, valor_compra, quantidade, status e canal.”

## ETAPA 3 — Valores ausentes

In [7]:
# Conte quantos valores ausentes existem por coluna
ausentes = df.isna().sum()
display(ausentes)

,0
pedido_id,0
cliente,1
data_pedido,0
categoria,0
valor_compra,2
quantidade,1
canal,0
status,0


In [10]:
# Calcule o percentual de ausentes
percentuais = (
    df.isna().mean() * 100
).round(2)

resumo_ausentes = pd.DataFrame({
    "quantidade": ausentes,
    "percentual": percentuais
})
display(resumo_ausentes)

,quantidade,percentual
pedido_id,0,0.00
cliente,1,2.63
data_pedido,0,0.00
categoria,0,0.00
valor_compra,2,5.26
quantidade,1,2.63
canal,0,0.00
status,0,0.00


In [11]:
display(
    df.loc[
        df["valor_compra"].isna(),
        ["pedido_id", "valor_compra"]
    ]
)

,pedido_id,valor_compra
8,P0009,NaN
24,P0025,NaN


### Cuidado importante
`NaN` não é o único tipo de ausência possível.

Uma célula pode conter:
- `NaN`
- texto vazio `""`
- apenas espaços `"   "`

Por isso, o mini-projeto pede também atenção a textos vazios.

## Investigar textos vazios e preenchidos apenas com espaços

In [12]:
clientes_em_branco = (
    df["cliente"]
    .astype("string")
    .str.strip()
    .eq("")
    .fillna(False)
)

print(
    "Clientes preenchidos apenas com espoços: ",
    clientes_em_branco.sum()
)

display(
    df.loc[
        clientes_em_branco,
        ["pedido_id", "cliente"]
    ]
)

Clientes preenchidos apenas com espoços:  1


,pedido_id,cliente
19,P0020,


In [14]:
# Combinação das duas situações

clientes_em_branco = (
    df["cliente"].isna() | clientes_em_branco
)

print(clientes_em_branco.sum())

2


## Duplicatas - Investigação de duplicidades e distinção de linhas de pedidos

Dois pedidos do mesmo cliente são automaticamente duplicados?

In [15]:
print(
    "Repetições completas além da primeira ocorrência: ",
    df.duplicated().sum() # marca as repetições da primeira ocorrência
)

print(
    "Linhas que participam de grupos repetidos: ",
    df.duplicated(keep=False).sum() # marca todas as ocorrências dos registros repetidos
)

print(
    "Identificadores de pedidos distintos: ",
    df["pedido_id"].nunique()
)

Repetições completas além da primeira ocorrência:  2
Linhas que participam de grupos repetidos:  4
Identificadores de pedidos distintos:  36


In [16]:
# Visualize as linhas duplicadas
display(
    df.loc[df.duplicated(keep=False)]
)

,pedido_id,cliente,data_pedido,categoria,valor_compra,quantidade,canal,status
3,P0004,Cliente 04,2026-09-25,Beleza,685.40,3.0,Site,entregue
7,P0008,Cliente 08,2026-09-06,Eletrônicos,173.53,4.0,App,entregue
36,P0004,Cliente 04,2026-09-25,Beleza,685.40,3.0,Site,entregue
37,P0008,Cliente 08,2026-09-06,Eletrônicos,173.53,4.0,App,entregue


### Pergunta para pensar
Dois pedidos do mesmo cliente são automaticamente duplicatas?

**Não.**  
Duplicata é um registro repetido de forma indevida. Um mesmo cliente pode fazer várias compras legítimas.

## ETAPA 5 — Inconsistências de texto - Investigação de categorias, canais e status

In [17]:
display(
    df["categoria"].value_counts(dropna=False)
)

,count
categoria,
Beleza,13
Moda,11
Eletrônicos,7
Casa,5
eletronicos,1
ELETRÔNICOS,1


In [18]:
display(
    df["canal"].value_counts(dropna=False)
)

,count
canal,
Loja,16
Site,12
App,8
app,1
SITE,1


In [19]:
display(
  df["status"].value_counts(dropna=False)
)

,count
status,
entregue,26
pendente,9
cancelado,3


- 1. Quantas linhas e colunas existem?

A base possui 38 linhas e 8 colunas


- 2. O que cada registro representa?

Cada registro descreve um pedido, com informações sobre o cliente, a data, a categoria, o valor, quantidade, e o canal e a situação do pedido.


- 3. Quais informações estão ausentes?

O diagnóstico isna() identificou:

Cliente - 1 valor ausente, Pedido envolvido P0005

valor_compra - 2 valores ausentes, Pedidos P009 e P0025

quantidade - 1 valor ausente, Pedido P0014

Ou seja, existem quatro células com valores ausentes, distribuídas em três colunas. As demais colunas, não apresentam ausências reconhecidas por ESSA verificação.

- 4. Existem textos preenchidos apenas com espaços?

Sim. No pedido P0020, a coluna cliente contém apenas espaços " ". Esse conteúdo não é identificado pelo isna(), porque existem caracteres armazenados na célula.

Considerando o cliente ausente de P0005 e o cliente preenchido apenas com espaços de P0020, existem dois registros sem um nome de cliente utilizável.

- 5. Quantas repetições completas excedentes existem?

Duas repetições completas excedentes:

O pedido P0004 que aparece duas vezes; E o pedido P0008 que também aparece duas vezes.


6. Quais diferenças de escrita foram encontras?

Na coluna categoria:
Eletrônicos - com forma inicial maiúscula e acento; eletronicos - com letras minusculas e ausêncioa de acento; ELETRÔNICOS - com letras maiúsculas e espaço final.

Coluna canal:

App e app - Maiúscula/minúscula e o espaço no início

Site e SITE - Maiúsculas e espaço no final.

- 7. Quais decisões ainda precisam ser identificadas?

Como tratar as ausências;

Como tratar o cliente preenchido com espaços;

Quais repetições remover;

Como padronizar os textos;

Quais registros consideramos nas análises.

- Escolha um problema encontrado, explicando como ele pode afetar a análise e propor uma decisão de tratamento.

Problema escolhido: pedidos repetidos

Problema encontrado: Os pedidos P0004 e P0008 aparecem duas vezes cada um, com todas as informações iguais.

Como afeta a análise: as repetições elas podem aumentar indevidamente a contagem de pedidos, e a soma de valores de compra, produzindo resultados superiores aos registrados sem essas cópias.

Decisão de tratamento proposta: após confirmar que são repetições indevidas, manter UMA ocorrência de cada pedido e remover as duas cópias excedentes. Somente essa operação reduziria a base de 38 para 36 linhas.

In [20]:
df.to_csv("pedidos_diagnosticos_limpos.csv",
          index=False)

In [22]:
files.download("pedidos_diagnosticos_limpos.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Interpretação
Para o computador, estes textos são diferentes:

- `Eletrônicos`
- `eletronicos`
- `ELETRÔNICOS `

Mesmo que semanticamente representem a mesma categoria.

Seu trabalho é **identificar primeiro**; a correção será praticada na aula de 22/10.

## DESAFIO FINAL — RELATÓRIO DE DIAGNÓSTICO

Escreva uma célula de texto respondendo:

1. Quantas linhas e colunas existem?
2. Qual é a unidade da análise?
3. Quais colunas possuem ausências?
4. Existem textos vazios?
5. Quantas duplicatas exatas foram encontradas?
6. Quais inconsistências textuais foram observadas?
7. O que você **ainda não alteraria** sem investigar melhor?

### Entrega de treino
Uma célula Markdown com um diagnóstico de 5 a 8 linhas.